# 01 – Exploring Steam reviews
Goal: get a feel for the data before choosing an NLP approach.

In [1]:
import sqlite3
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 300)

# Path is relative to the notebooks/ folder
conn = sqlite3.connect("../data/reviews.db")
df = pd.read_sql("SELECT * FROM reviews", conn)

df["hours_at_review"] = df["playtime_at_review"].fillna(0) / 60
df["length"] = df["text"].str.len()
df["created"] = pd.to_datetime(df["created_at"], unit="s")
df.head()

,review_id,appid,language,text,voted_up,votes_up,votes_funny,weighted_vote_score,playtime_at_review,playtime_forever,early_access,received_for_free,created_at,updated_at,hours_at_review,length,created
0,235632430,1086940,english,"My favourite game, never gets old",1,0,0,0.500000,458,488,0,0,1789844177,1789844177,7.633333,33,2026-09-19 18:56:17
1,235631900,1086940,english,"Did not really know what to expect, but pleasantly surprised. Overwhelmingly complex if all aspects are to be understood, but can also be played and enjoyed with a simple approach.",1,0,0,0.500000,7642,7642,0,0,1789843703,1789843703,127.366667,180,2026-09-19 18:48:23
2,235630492,1086940,english,I love the graphics and the storyline.,1,0,0,0.500000,25792,25852,0,0,1789842497,1789842497,429.866667,38,2026-09-19 18:28:17
3,235629522,1086940,english,Greatest game over,1,0,0,0.500000,10170,10182,0,0,1789841656,1789841656,169.500000,18,2026-09-19 18:14:16
4,235629211,1086940,english,Terrible tutorial bad movement system not friendly to new players honestly need to get a better tutorial I shouldn't have to google every quest thats just bad game design. Clicking simulator!!!!!!!!!!,0,0,0,0.435767,287,287,0,0,1789841389,1789841514,4.783333,200,2026-09-19 18:09:49


## Overview

In [2]:
print(f"{len(df)} reviews")
print(f"Positive: {df['voted_up'].mean():.1%}")
print("\nReview length (characters):")
print(df["length"].describe().round(0))
print("\nHours played at review time:")
print(df["hours_at_review"].describe().round(1))

5098 reviews
Positive: 94.8%

Review length (characters):
count    5098.0
mean      186.0
std       495.0
min         0.0
25%        15.0
50%        48.0
75%       147.0
max      7996.0
Name: length, dtype: float64

Hours played at review time:
count     5098.0
mean       240.5
std        364.8
min          0.1
25%         48.4
50%        134.6
75%        288.0
max      11001.5
Name: hours_at_review, dtype: float64


## Negative reviews worth reading
Short reviews are mostly jokes or one-liners ("no", "10/10 would crash again"), so we filter them out.

In [3]:
neg = df[(df["voted_up"] == 0) & (df["length"] >= 50)].copy()
print(f"{len(neg)} substantial negative reviews")

for _, r in neg.sample(min(20, len(neg)), random_state=42).iterrows():
    print(f"--- {r['hours_at_review']:.1f}h played ---")
    print(r["text"][:600])
    print()

198 substantial negative reviews
--- 6.3h played ---
It would be nice if the RNG goddess hadn't forsaken me and I could just actually play the damn game. I hate LUCK based combat as I have been proven to have the absolute worst luck in the universe. If you like Fallout 1 & 2's combat style you're going to have a good time. Otherwise prepare thine bootyhole for a bad day at a gay bar called Gimplies Goonhole on a day you forgot to bring lube while blindfolded.

--- 79.4h played ---
decent story but combat system makes it extremely tedious
way too many enemies for this turnbased system, each round takes 40 minutes to complete.. i browsed reddit more than i played during my co-op campaign with friends

--- 162.8h played ---
The final fight is the worst fight of any final game I've gotten to play. It is slow and unfun, the fight consists of about a dozen enemies with abilities to skip your turn, you can go the entire fight not having moved once, its ♥♥♥♥♥♥♥ insane to me that such an amazin

## Most helpful complaints
Reviews the community upvoted most are a good proxy for widely shared pain points.

In [4]:
neg.sort_values("votes_up", ascending=False)[["votes_up", "hours_at_review", "text"]].head(10)

,votes_up,hours_at_review,text
1585,58,38.800000,"I seem to be in the rare minority of people who just can't get into this game. \n\nNearly 40 hours put into the game and I can't get through the first Act, not because of difficulty, but because I just don't like the characters or the story enough to care about any of it. The writing for charact..."
4269,51,30.150000,"I genuinely tried to like Baldur’s Gate 3. I loved the first two games, and after hearing so much praise for this one, I gave it 27 hours across two separate playthroughs. Unfortunately, I think I am finished with it.\r\n\r\nThe presentation is excellent. The graphics, music and voice acting are..."
2699,36,22.400000,"After trying the game for a second time, I still have no idea what is supposed to be fun about it.\n\nI can't stand the graphics, character creation is terrible, dialogue is crap, most characters can be boiled down to ""Look at me, I'm so edgy, isn't it cool how edgy I am?"", the combat system is ..."
1084,33,87.766667,"Honestly, I've had this game since release and I just can't with it. All of the characters are so ♥♥♥♥♥♥♥ cringe I cannot stand the story long enough to make it past the first act. \n\nIt's like being stuck with a bunch of hypersexual theater kids all doing amateur auditions and thinking they're..."
3759,20,9.633333,"Highly Rated, But a Tedious and Punishing Trial-and-Error Experience\r\nRating: Not Recommended\r\nIf you are buying this game based on its glowing masterwork reviews, hoping for a smooth and immersive fantasy RPG adventure to relax with after work, you should think twice.\r\n1. Extremely Vague ..."
4656,16,558.000000,"Play divinity original sin 2 instead, the npc's treat you like ♥♥♥♥ in both games, and dos2 has a more compelling story to this game, which is ironic because this game is not Baldur's Gate 3 It is Divinity original sin 3, with a very well crafted veneer of Faerun on top. You have to mod this gam..."
4689,12,907.283333,"Extremely overrated. The writing is lazy and terrible (shipwrecked with lapse memory, again) and most of the choices presented make no sense (let your family die or suffer 2 seconds of pain occasionally...). The story has nearly nothing to do with the BG1+2 even if you go Dark Urge so why name i..."
3038,12,27.516667,"Nearly perfect for what it is, but people like me might just be the wrong audience. While I love the mature story and freedom to approach the game in a vast amount of ways, the fundamentals are lacking--slow, repetitive, cumbersome user interface and game mechanics. The game is wonderful for the..."
2788,12,10.116667,Where to begin?\n\n1. Laziest storytelling. The game treats you like you already know everything or at least care. If you don't - the game doesn't even try to immerse you. \n\n2. Bland characters that do not act or talk like real people would have. They dump their exposition on you with tons of ...
4072,12,26.233333,"Baldur's Gate 3 is a turn-based role-playing game. The game has high-quality graphics, but the narrative destroys absolute moral values. The story replaces traditional virtue with modern moral relativism.\n\nThe narrative does not show absolute objective truth or sacred order. Companions reject ..."


## First signal: which words are distinctive to negative reviews?
A simple baseline: compare how often each word/phrase appears in negative vs positive reviews (log-odds ratio). No ML yet.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

subset = df[df["length"] >= 50]
vec = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5)
X = vec.fit_transform(subset["text"])
terms = vec.get_feature_names_out()

is_neg = (subset["voted_up"] == 0).to_numpy()
neg_counts = np.asarray(X[is_neg].sum(axis=0)).ravel()
pos_counts = np.asarray(X[~is_neg].sum(axis=0)).ravel()

log_odds = (np.log((neg_counts + 1) / (neg_counts.sum() + len(terms)))
            - np.log((pos_counts + 1) / (pos_counts.sum() + len(terms))))

result = pd.DataFrame({"term": terms, "neg": neg_counts, "pos": pos_counts, "log_odds": log_odds})
result[result["neg"] >= 10].sort_values("log_odds", ascending=False).head(30)

## Notes
Write down what you notice here: recurring themes, jokes/noise, reviews mentioning several issues at once. These notes will guide the next step.